In [4]:
# ============================================================
# CREDRESOLVE — COLLECTIONS RECOVERY ANALYTICS
# 05 — DRIVER ANALYSIS
# ============================================================
#
# Purpose:
# Identify factors associated with collections recovery.
#
# IMPORTANT:
# Association is NOT treated as causation.
# Raw data is never modified.
# Golden tables are used as the analytical source.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

GOLDEN_DIR = PROJECT_ROOT / "data" / "golden"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 90)
print("CREDRESOLVE — DRIVER ANALYSIS")
print("=" * 90)


# ------------------------------------------------------------
# 2. LOAD GOLDEN TABLES
# ------------------------------------------------------------

golden_files = sorted(
    GOLDEN_DIR.glob("*_golden.csv")
)

golden = {
    file.stem.replace("_golden", ""):
        pd.read_csv(file)
    for file in golden_files
}

print(
    f"Golden datasets loaded: {len(golden)}"
)


# ------------------------------------------------------------
# 3. CHECK REQUIRED TABLES
# ------------------------------------------------------------

required_tables = [
    "accounts",
    "payments",
    "calls",
    "call_attempts",
    "call_dispositions",
    "agents",
    "agent_sessions",
    "daily_targeting",
    "campaigns",
    "vendor_telephony"
]

missing_tables = [
    table
    for table in required_tables
    if table not in golden
]

if missing_tables:

    print(
        "Missing Golden tables:",
        missing_tables
    )

    raise ValueError(
        "Required Golden tables are missing."
    )

print("All required Golden tables are available.")


# ============================================================
# 4. PREPARE ACCOUNT-LEVEL ANALYTICAL BASE
# ============================================================

accounts = golden["accounts"].copy()

# Standardized date
accounts["opened_at_standardized"] = pd.to_datetime(
    accounts["opened_at_standardized"],
    errors="coerce"
)

accounts["analysis_month"] = (
    accounts["opened_at_standardized"]
    .dt.to_period("M")
    .astype("string")
)

# ------------------------------------------------------------
# Payment outcomes
# ------------------------------------------------------------

payments = golden["payments"].copy()

payments["event_at_standardized"] = pd.to_datetime(
    payments["event_at_standardized"],
    errors="coerce"
)

# Numeric payment amount
payments["amount"] = pd.to_numeric(
    payments["amount"],
    errors="coerce"
)

# Treat completed/success-like statuses as recovery
payment_status = (
    payments["payment_status"]
    .astype("string")
    .str.lower()
    .str.strip()
)

successful_statuses = {
    "success",
    "successful",
    "completed",
    "paid",
    "settled"
}

payments["successful_payment_flag"] = (
    payment_status.isin(
        successful_statuses
    )
)

# ------------------------------------------------------------
# Account-level payment metrics
# ------------------------------------------------------------

payment_account_metrics = (
    payments
    .groupby("account_id")
    .agg(
        payment_events=(
            "payment_id",
            "nunique"
        ),
        successful_payments=(
            "successful_payment_flag",
            "sum"
        ),
        total_payment_amount=(
            "amount",
            "sum"
        ),
        payment_references=(
            "payment_reference",
            "nunique"
        )
    )
    .reset_index()
)

payment_account_metrics[
    "recovered_flag"
] = (
    payment_account_metrics[
        "successful_payments"
    ] > 0
)

# ------------------------------------------------------------
# Merge account + payment information
# ------------------------------------------------------------

driver_base = accounts.merge(
    payment_account_metrics,
    on="account_id",
    how="left"
)

for column in [
    "payment_events",
    "successful_payments",
    "total_payment_amount",
    "payment_references"
]:

    driver_base[column] = (
        driver_base[column]
        .fillna(0)
    )

driver_base["recovered_flag"] = (
    driver_base["recovered_flag"]
    .fillna(False)
)


# ============================================================
# 5. CALL / CONTACT FEATURES
# ============================================================

calls = golden["calls"].copy()

calls["event_at_standardized"] = pd.to_datetime(
    calls["event_at_standardized"],
    errors="coerce"
)

# Calls per account
call_account_metrics = (
    calls
    .groupby("account_id")
    .agg(
        total_calls=(
            "call_id",
            "nunique"
        ),
        unique_agents=(
            "agent_id",
            "nunique"
        ),
        unique_campaigns=(
            "campaign_id",
            "nunique"
        ),
        unique_vendors=(
            "vendor_id",
            "nunique"
        ),
        total_call_duration_sec=(
            "duration_sec",
            "sum"
        )
    )
    .reset_index()
)

driver_base = driver_base.merge(
    call_account_metrics,
    on="account_id",
    how="left"
)

for column in [
    "total_calls",
    "unique_agents",
    "unique_campaigns",
    "unique_vendors",
    "total_call_duration_sec"
]:

    driver_base[column] = (
        driver_base[column]
        .fillna(0)
    )


# ------------------------------------------------------------
# Contact / call status features
# ------------------------------------------------------------

call_status_profile = (
    calls
    .assign(
        call_status_clean=
        calls["call_status"]
        .astype("string")
        .str.lower()
        .str.strip()
    )
    .groupby(
        [
            "account_id",
            "call_status_clean"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reset_index()
)

# Rename status-derived columns
status_columns = [
    column
    for column in call_status_profile.columns
    if column != "account_id"
]

for column in status_columns:

    safe_name = (
        "call_status_"
        +
        str(column)
        .replace(" ", "_")
        .replace("/", "_")
    )

    call_status_profile = (
        call_status_profile
        .rename(
            columns={
                column:
                safe_name
            }
        )
    )

driver_base = driver_base.merge(
    call_status_profile,
    on="account_id",
    how="left"
)


# ============================================================
# 6. TARGETING / CAMPAIGN FEATURES
# ============================================================

targeting = golden["daily_targeting"].copy()

targeting["target_date_standardized"] = pd.to_datetime(
    targeting["target_date_standardized"],
    errors="coerce"
)

targeting_metrics = (
    targeting
    .groupby("account_id")
    .agg(
        targeting_events=(
            "target_id",
            "nunique"
        ),
        campaigns_targeted=(
            "campaign_id",
            "nunique"
        ),
        targeting_priorities=(
            "priority",
            "nunique"
        ),
        targeting_statuses=(
            "status",
            "nunique"
        )
    )
    .reset_index()
)

driver_base = driver_base.merge(
    targeting_metrics,
    on="account_id",
    how="left"
)

for column in [
    "targeting_events",
    "campaigns_targeted",
    "targeting_priorities",
    "targeting_statuses"
]:

    driver_base[column] = (
        driver_base[column]
        .fillna(0)
    )


# ============================================================
# 7. ATTEMPT / DISPOSITION FEATURES
# ============================================================

attempts = golden["call_attempts"].copy()

attempt_metrics = (
    attempts
    .groupby("account_id")
    .agg(
        total_attempts=(
            "attempt_id",
            "nunique"
        ),
        max_attempt_number=(
            "attempt_no",
            "max"
        ),
        attempt_statuses=(
            "attempt_status",
            "nunique"
        )
    )
    .reset_index()
)

driver_base = driver_base.merge(
    attempt_metrics,
    on="account_id",
    how="left"
)

for column in [
    "total_attempts",
    "max_attempt_number",
    "attempt_statuses"
]:

    driver_base[column] = (
        driver_base[column]
        .fillna(0)
    )


# ------------------------------------------------------------
# Disposition metrics
# ------------------------------------------------------------

dispositions = golden[
    "call_dispositions"
].copy()

disposition_metrics = (
    dispositions
    .groupby("account_id")
    .agg(
        disposition_events=(
            "disposition_id",
            "nunique"
        ),
        disposition_codes=(
            "disposition_code",
            "nunique"
        ),
        disposition_versions=(
            "disposition_version",
            "nunique"
        )
    )
    .reset_index()
)

driver_base = driver_base.merge(
    disposition_metrics,
    on="account_id",
    how="left"
)

for column in [
    "disposition_events",
    "disposition_codes",
    "disposition_versions"
]:

    driver_base[column] = (
        driver_base[column]
        .fillna(0)
    )


# ============================================================
# 8. AGENT / VENDOR FEATURES
# ============================================================

agent_account_metrics = (
    calls
    .groupby("account_id")
    .agg(
        agent_count=(
            "agent_id",
            "nunique"
        ),
        vendor_count=(
            "vendor_id",
            "nunique"
        )
    )
    .reset_index()
)

driver_base = driver_base.merge(
    agent_account_metrics,
    on="account_id",
    how="left"
)

driver_base[
    "agent_count"
] = driver_base[
    "agent_count"
].fillna(0)

driver_base[
    "vendor_count"
] = driver_base[
    "vendor_count"
].fillna(0)


# ============================================================
# 9. RECOVERY RATE BY PORTFOLIO SEGMENT
# ============================================================

print("\n" + "=" * 90)
print("RECOVERY BY PORTFOLIO SEGMENT")
print("=" * 90)

segment_columns = [
    "loan_type",
    "risk_segment",
    "status"
]

segment_results = []

for column in segment_columns:

    result = (
        driver_base
        .groupby(
            column,
            dropna=False
        )
        .agg(
            accounts=(
                "account_id",
                "nunique"
            ),
            recovered_accounts=(
                "recovered_flag",
                "sum"
            ),
            total_recovery=(
                "total_payment_amount",
                "sum"
            ),
            avg_dpd=(
                "dpd",
                "mean"
            ),
            avg_outstanding=(
                "outstanding_amount",
                "mean"
            )
        )
        .reset_index()
    )

    result["recovery_rate"] = (
        result["recovered_accounts"]
        /
        result["accounts"]
    )

    result["driver_dimension"] = column

    segment_results.append(
        result
    )

portfolio_driver_df = pd.concat(
    segment_results,
    ignore_index=True
)

display(portfolio_driver_df)

portfolio_driver_df.to_csv(
    OUTPUT_DIR /
    "driver_portfolio_segments.csv",
    index=False
)


# ============================================================
# 10. DPD BUCKET ANALYSIS
# ============================================================

def dpd_bucket(value):

    if pd.isna(value):
        return "Missing"

    if value <= 0:
        return "Current"

    if value <= 30:
        return "1-30"

    if value <= 60:
        return "31-60"

    if value <= 90:
        return "61-90"

    return "90+"


driver_base["dpd_bucket"] = (
    driver_base["dpd"]
    .apply(dpd_bucket)
)

dpd_analysis = (
    driver_base
    .groupby("dpd_bucket")
    .agg(
        accounts=(
            "account_id",
            "nunique"
        ),
        recovered_accounts=(
            "recovered_flag",
            "sum"
        ),
        recovery_amount=(
            "total_payment_amount",
            "sum"
        ),
        avg_outstanding=(
            "outstanding_amount",
            "mean"
        )
    )
    .reset_index()
)

dpd_analysis["recovery_rate"] = (
    dpd_analysis["recovered_accounts"]
    /
    dpd_analysis["accounts"]
)

print("\n" + "=" * 90)
print("RECOVERY BY DPD BUCKET")
print("=" * 90)

display(dpd_analysis)

dpd_analysis.to_csv(
    OUTPUT_DIR /
    "driver_dpd_analysis.csv",
    index=False
)


# ============================================================
# 11. RISK SEGMENT ANALYSIS
# ============================================================

risk_analysis = (
    driver_base
    .groupby("risk_segment", dropna=False)
    .agg(
        accounts=(
            "account_id",
            "nunique"
        ),
        recovered_accounts=(
            "recovered_flag",
            "sum"
        ),
        recovery_amount=(
            "total_payment_amount",
            "sum"
        ),
        avg_dpd=(
            "dpd",
            "mean"
        ),
        avg_outstanding=(
            "outstanding_amount",
            "mean"
        )
    )
    .reset_index()
)

risk_analysis["recovery_rate"] = (
    risk_analysis["recovered_accounts"]
    /
    risk_analysis["accounts"]
)

print("\n" + "=" * 90)
print("RECOVERY BY RISK SEGMENT")
print("=" * 90)

display(risk_analysis)

risk_analysis.to_csv(
    OUTPUT_DIR /
    "driver_risk_analysis.csv",
    index=False
)


# ============================================================
# 12. CHANNEL / CAMPAIGN ANALYSIS
# ============================================================

campaigns = golden["campaigns"].copy()

campaign_lookup = campaigns[
    [
        "campaign_id",
        "campaign_name",
        "channel",
        "strategy_version"
    ]
].drop_duplicates("campaign_id")

calls_campaign = calls.merge(
    campaign_lookup,
    on="campaign_id",
    how="left"
)

campaign_account = (
    calls_campaign
    [
        [
            "account_id",
            "campaign_id",
            "campaign_name",
            "channel",
            "strategy_version"
        ]
    ]
    .drop_duplicates()
)

campaign_account = campaign_account.merge(
    driver_base[
        [
            "account_id",
            "recovered_flag",
            "total_payment_amount"
        ]
    ],
    on="account_id",
    how="left"
)

campaign_analysis = (
    campaign_account
    .groupby(
        [
            "campaign_id",
            "campaign_name",
            "channel",
            "strategy_version"
        ],
        dropna=False
    )
    .agg(
        accounts=(
            "account_id",
            "nunique"
        ),
        recovered_accounts=(
            "recovered_flag",
            "sum"
        ),
        recovery_amount=(
            "total_payment_amount",
            "sum"
        )
    )
    .reset_index()
)

campaign_analysis["recovery_rate"] = (
    campaign_analysis["recovered_accounts"]
    /
    campaign_analysis["accounts"]
)

print("\n" + "=" * 90)
print("RECOVERY BY CAMPAIGN / CHANNEL")
print("=" * 90)

display(campaign_analysis)

campaign_analysis.to_csv(
    OUTPUT_DIR /
    "driver_campaign_analysis.csv",
    index=False
)


# ============================================================
# 13. AGENT-LEVEL ANALYSIS
# ============================================================

agent_account = (
    calls
    [
        [
            "account_id",
            "agent_id"
        ]
    ]
    .drop_duplicates()
)

agent_account = agent_account.merge(
    driver_base[
        [
            "account_id",
            "recovered_flag",
            "total_payment_amount"
        ]
    ],
    on="account_id",
    how="left"
)

agent_analysis = (
    agent_account
    .groupby("agent_id", dropna=False)
    .agg(
        accounts=(
            "account_id",
            "nunique"
        ),
        recovered_accounts=(
            "recovered_flag",
            "sum"
        ),
        recovery_amount=(
            "total_payment_amount",
            "sum"
        )
    )
    .reset_index()
)

agent_analysis["recovery_rate"] = (
    agent_analysis["recovered_accounts"]
    /
    agent_analysis["accounts"]
)

print("\n" + "=" * 90)
print("AGENT-LEVEL RECOVERY ANALYSIS")
print("=" * 90)

display(
    agent_analysis.sort_values(
        "recovery_rate",
        ascending=False
    ).head(30)
)

agent_analysis.to_csv(
    OUTPUT_DIR /
    "driver_agent_analysis.csv",
    index=False
)


# ============================================================
# 14. CORRELATION / NUMERIC DRIVER SCREEN
# ============================================================

numeric_driver_columns = [
    "dpd",
    "principal_amount",
    "outstanding_amount",
    "total_calls",
    "total_attempts",
    "total_call_duration_sec",
    "targeting_events",
    "campaigns_targeted",
    "disposition_events",
    "agent_count",
    "vendor_count"
]

available_numeric = [
    column
    for column in numeric_driver_columns
    if column in driver_base.columns
]

correlation_rows = []

for column in available_numeric:

    subset = driver_base[
        [
            column,
            "recovered_flag"
        ]
    ].dropna()

    if (
        len(subset) > 1
        and subset[column].nunique() > 1
    ):

        correlation = (
            subset[column]
            .corr(
                subset["recovered_flag"]
            )
        )

    else:

        correlation = np.nan

    correlation_rows.append({
        "driver": column,
        "pearson_correlation_with_recovery":
            correlation,
        "observations":
            len(subset)
    })

correlation_df = pd.DataFrame(
    correlation_rows
)

correlation_df[
    "absolute_correlation"
] = correlation_df[
    "pearson_correlation_with_recovery"
].abs()

correlation_df = correlation_df.sort_values(
    "absolute_correlation",
    ascending=False
)

print("\n" + "=" * 90)
print("NUMERIC DRIVER SCREEN")
print("=" * 90)

display(correlation_df)

correlation_df.to_csv(
    OUTPUT_DIR /
    "driver_numeric_correlations.csv",
    index=False
)


# ============================================================
# 15. EXPOSURE / RECOVERY RELATIONSHIP
# ============================================================

driver_base["calls_per_account"] = (
    driver_base["total_calls"]
)

driver_base["attempts_per_account"] = (
    driver_base["total_attempts"]
)

exposure_bands = pd.qcut(
    driver_base["total_calls"],
    q=4,
    duplicates="drop"
)

exposure_analysis = (
    driver_base
    .groupby(
        exposure_bands,
        observed=False
    )
    .agg(
        accounts=(
            "account_id",
            "nunique"
        ),
        recovered_accounts=(
            "recovered_flag",
            "sum"
        ),
        recovery_amount=(
            "total_payment_amount",
            "sum"
        ),
        avg_dpd=(
            "dpd",
            "mean"
        )
    )
    .reset_index()
)

exposure_analysis["recovery_rate"] = (
    exposure_analysis["recovered_accounts"]
    /
    exposure_analysis["accounts"]
)

print("\n" + "=" * 90)
print("RECOVERY BY CALL EXPOSURE")
print("=" * 90)

display(exposure_analysis)

exposure_analysis.to_csv(
    OUTPUT_DIR /
    "driver_exposure_analysis.csv",
    index=False
)


# ============================================================
# 16. DRIVER ANALYTICAL DATASET
# ============================================================

driver_base.to_csv(
    OUTPUT_DIR /
    "driver_analysis_account_level.csv",
    index=False
)


# ============================================================
# 17. DRIVER SUMMARY
# ============================================================

summary = pd.DataFrame([{

    "accounts_analyzed":
        driver_base["account_id"].nunique(),

    "recovered_accounts":
        int(
            driver_base["recovered_flag"].sum()
        ),

    "overall_recovery_rate":
        driver_base["recovered_flag"].mean(),

    "total_recovery_amount":
        driver_base[
            "total_payment_amount"
        ].sum(),

    "driver_dimensions_analyzed":
        len(segment_columns),

    "numeric_drivers_screened":
        len(available_numeric),

    "campaigns_analyzed":
        campaign_analysis[
            "campaign_id"
        ].nunique(),

    "agents_analyzed":
        agent_analysis[
            "agent_id"
        ].nunique()
}])

print("\n" + "=" * 90)
print("DRIVER ANALYSIS SUMMARY")
print("=" * 90)

display(summary)

summary.to_csv(
    OUTPUT_DIR /
    "driver_analysis_summary.csv",
    index=False
)


# ============================================================
# 18. FINAL STATUS
# ============================================================

print("\n" + "=" * 90)
print("DRIVER ANALYSIS COMPLETE")
print("=" * 90)

print(
    "Golden Dataset was used as the analytical source."
)

print(
    "Raw source files were NOT modified."
)

print(
    "Driver results represent associations, "
    "not causal effects."
)

print(
    f"Driver outputs saved to: {OUTPUT_DIR}"
)

CREDRESOLVE — DRIVER ANALYSIS
Golden datasets loaded: 18
All required Golden tables are available.


C:\Users\DELL\AppData\Local\Temp\ipykernel_18772\2218403179.py:211: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)



RECOVERY BY PORTFOLIO SEGMENT


,loan_type,accounts,recovered_accounts,total_recovery,avg_dpd,avg_outstanding,recovery_rate,driver_dimension,risk_segment,status
0,AUTO,6079,2700,3.889808e+08,56.451390,351258.569006,0.444152,loan_type,NaN,NaN
1,BNPL,5928,2580,3.720065e+08,56.381073,347320.281665,0.435223,loan_type,NaN,NaN
2,CONSUMER,5930,2685,3.867460e+08,56.611636,349597.418543,0.452782,loan_type,NaN,NaN
3,CREDIT_CARD,6080,2690,3.895328e+08,56.731250,349782.391488,0.442434,loan_type,NaN,NaN
4,PERSONAL,5983,2629,3.799925e+08,56.353836,350163.837555,0.439412,loan_type,NaN,NaN
5,NaN,7552,3300,4.816205e+08,56.447961,350394.940613,0.436970,risk_segment,HIGH,NaN
6,NaN,7513,3381,4.868134e+08,57.972048,350500.611721,0.450020,risk_segment,LOW,NaN
7,NaN,7533,3335,4.765620e+08,54.763972,348888.816837,0.442719,risk_segment,MEDIUM,NaN
8,NaN,7402,3268,4.722628e+08,56.851797,348738.475871,0.441502,risk_segment,NPA,NaN
9,NaN,7539,3315,4.807688e+08,56.257992,349833.775859,0.439713,status,NaN,ACTIVE



RECOVERY BY DPD BUCKET


...


RECOVERY BY CALL EXPOSURE


,total_calls,accounts,recovered_accounts,recovery_amount,avg_dpd,recovery_rate
0,"(-0.001, 2.0]",12637,5547,8.029306e+08,56.266282,0.438949
1,"(2.0, 3.0]",6785,2983,4.379185e+08,56.946647,0.439646
2,"(3.0, 4.0]",5030,2259,3.219432e+08,57.316700,0.449105
3,"(4.0, 13.0]",5548,2495,3.544663e+08,55.780461,0.449712



DRIVER ANALYSIS SUMMARY


,accounts_analyzed,recovered_accounts,overall_recovery_rate,total_recovery_amount,driver_dimensions_analyzed,numeric_drivers_screened,campaigns_analyzed,agents_analyzed
0,30000,13284,0.4428,1.917259e+09,3,11,120,1000



DRIVER ANALYSIS COMPLETE
Golden Dataset was used as the analytical source.
Raw source files were NOT modified.
Driver results represent associations, not causal effects.
Driver outputs saved to: c:\Users\DELL\Documents\CredResolve Collections Recovery Analytics\outputs\tables
